# Notebook 2 — Dynamic Programming and Value-Based RL

<div class="alert alert-success">

**Learning outcomes:**
By the end of this notebook you should be able to:
- state and apply the Bellman evaluation and optimality equations,
- implement value iteration for tabular MDPs,
- explain approximate value iteration and its connection to Q-learning,
- implement a Deep Q-Network (DQN) with experience replay and target networks,
- describe the key ideas of Soft Actor-Critic (SAC) for continuous action spaces.
</div>

# 1. Bellman Equations

## 1.1 Bellman evaluation equation

A key property of $v^\pi$ (and $q^\pi$) is that they satisfy a fixed-point equation.

<div class="alert alert-success">

**Bellman evaluation equation for $q^\pi$:**
$$q^\pi(s,a) = r(s,a) + \gamma \sum_{s'} p(s'|s,a) \sum_{a'} \pi(a'|s')\, q^\pi(s',a')$$

Or in operator notation: $q^\pi = \mathcal{T}^\pi q^\pi$, where $\mathcal{T}^\pi$ is the **Bellman evaluation operator**.
</div>

Intuition: the value of $(s,a)$ = immediate reward + discounted value of where we end up.

## 1.2 Bellman optimality equation

<div class="alert alert-success">

**Bellman optimality equation for $q^*$:**
$$q^*(s,a) = r(s,a) + \gamma \sum_{s'} p(s'|s,a) \max_{a'} q^*(s',a')$$

Or: $q^* = \mathcal{T}^* q^*$, where $\mathcal{T}^*$ is the **Bellman optimality operator**.
</div>

Both operators are **contractions** with modulus $\gamma < 1$, so iterating them from any starting point converges to the unique fixed point.

# 2. Value Iteration on FrozenLake

**Value iteration** repeatedly applies $\mathcal{T}^*$:
$$q_{n+1} \leftarrow \mathcal{T}^* q_n, \quad q_0 = 0.$$

Convergence is guaranteed: $\|q_n - q^*\|_\infty \leq \gamma^n \|q_0 - q^*\|_\infty$.

In [ ]:
import gymnasium as gym
import gymnasium.envs.toy_text.frozen_lake as fl
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

env = gym.make('FrozenLake-v1', render_mode="ansi")
env.reset()

n_states  = env.observation_space.n
n_actions = env.action_space.n
gamma = 0.99

In [ ]:
def value_iteration(env, gamma, tol=1e-6, max_iter=5000):
    """Tabular value iteration. Returns Q* and convergence residuals."""
    n_s = env.observation_space.n
    n_a = env.action_space.n
    Q = np.zeros((n_s, n_a))
    residuals = []

    for _ in range(max_iter):
        Q_new = np.zeros_like(Q)
        for s in range(n_s):
            for a in range(n_a):
                for prob, s_next, reward, done in env.unwrapped.P[s][a]:
                    if done:
                        Q_new[s, a] += prob * reward
                    else:
                        Q_new[s, a] += prob * (reward + gamma * np.max(Q[s_next]))
        residual = np.max(np.abs(Q_new - Q))
        residuals.append(residual)
        Q = Q_new
        if residual < tol:
            break

    return Q, np.array(residuals)

Q_star, residuals = value_iteration(env, gamma)
print(f"Converged in {len(residuals)} iterations, final residual {residuals[-1]:.2e}")

In [ ]:
# Visualise convergence
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.plot(residuals)
plt.xlabel("Iteration")
plt.ylabel("Max Bellman residual")
plt.title("Value iteration convergence")

plt.subplot(1, 2, 2)
plt.semilogy(residuals)
plt.xlabel("Iteration")
plt.ylabel("Residual (log scale)")
plt.tight_layout()
plt.show()

In [ ]:
# Extract optimal policy and show it
actions_sym = {fl.LEFT: '←', fl.DOWN: '↓', fl.RIGHT: '→', fl.UP: '↑'}
pi_star = np.argmax(Q_star, axis=1)

print("Optimal Q* (reshaped 4×4 per action is hard to show; showing V* = max_a Q*):")
print(np.max(Q_star, axis=1).reshape(4,4).round(3))
print()
print("Optimal policy π*:")
for row in range(4):
    print("  " + "  ".join(actions_sym[pi_star[row*4+col]] for col in range(4)))

In [ ]:
# Evaluate the policy found by value iteration
def run_policy(env, policy, n=10_000, horizon=200, gamma=0.99):
    returns = []
    for _ in range(n):
        s, _ = env.reset()
        G, disc = 0.0, 1.0
        for _ in range(horizon):
            a = int(policy[s])
            s, r, done, trunc, _ = env.step(a)
            G += disc * r
            disc *= gamma
            if done or trunc:
                break
        returns.append(G)
    return np.mean(returns), np.std(returns)

mean_ret, std_ret = run_policy(env, pi_star)
print(f"Optimal policy: mean return = {mean_ret:.4f} ± {std_ret:.4f}")

# 3. From Tabular to Approximate Value Iteration

Tabular value iteration requires visiting every $(s, a)$ pair — only feasible for small discrete spaces.

For continuous or large state spaces we use **function approximation**:
$$q(s, a; \theta) \approx q^*(s, a).$$

**Approximate Value Iteration (AVI)** turns each Bellman update into a supervised learning problem:

<div class="alert alert-success">

At iteration $n$, collect a dataset of targets $y_i = r_i + \gamma \max_{a'} q(s'_i, a'; \theta_n)$ and minimise:
$$L_n(\theta) = \frac{1}{|\mathcal{B}|} \sum_{(s,a,r,s') \in \mathcal{B}} \left( q(s,a;\theta) - y_i \right)^2$$
</div>

Two challenges compared to tabular VI:
1. **Correlated samples** — consecutive environment transitions are not iid.
2. **Non-stationary targets** — $y_i$ depends on $\theta_n$ which keeps changing.

Deep Q-Networks (DQN) address both.

# 4. Deep Q-Networks (DQN)

DQN (Mnih et al., 2013/2015) is AVI with neural network function approximation and two key tricks.

## 4.1 Experience Replay

Store transitions $(s, a, r, s', d)$ in a **replay buffer** and sample **random mini-batches** to break temporal correlations.

## 4.2 Target Network

Keep a **frozen copy** $\hat{q}(\cdot; \theta^-)$ of the network for computing targets.
Update $\theta^-$ periodically (copy $\theta$ every $C$ steps).
This stabilises training by making targets move more slowly.

## 4.3 $\epsilon$-greedy exploration

Take a random action with probability $\epsilon$ (decreasing over training), greedy action otherwise.
This ensures the replay buffer contains diverse transitions.

In [ ]:
import gymnasium as gym
env_cp = gym.make('CartPole-v1', render_mode="rgb_array")
print("CartPole state space :", env_cp.observation_space)
print("CartPole action space:", env_cp.action_space)

In [ ]:
import random
import torch
import numpy as np

class ReplayBuffer:
    """Fixed-size FIFO replay buffer returning torch Tensors."""
    def __init__(self, capacity, device):
        self.capacity = int(capacity)
        self.data = []
        self.index = 0
        self.device = device

    def append(self, s, a, r, s_next, done):
        if len(self.data) < self.capacity:
            self.data.append(None)
        self.data[self.index] = (s, a, r, s_next, done)
        self.index = (self.index + 1) % self.capacity

    def sample(self, batch_size):
        batch = random.sample(self.data, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (
            torch.FloatTensor(np.array(s)).to(self.device),
            torch.LongTensor(np.array(a)).to(self.device),
            torch.FloatTensor(np.array(r)).to(self.device),
            torch.FloatTensor(np.array(s2)).to(self.device),
            torch.FloatTensor(np.array(d)).to(self.device),
        )

    def __len__(self):
        return len(self.data)

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def make_dqn(state_dim, n_actions, nb_neurons=128):
    return nn.Sequential(
        nn.Linear(state_dim, nb_neurons), nn.ReLU(),
        nn.Linear(nb_neurons, nb_neurons), nn.ReLU(),
        nn.Linear(nb_neurons, n_actions)
    ).to(device)

state_dim = env_cp.observation_space.shape[0]
n_actions = env_cp.action_space.n
model = make_dqn(state_dim, n_actions)
print(model)

In [ ]:
import copy
from tqdm import trange

def greedy_action(network, state):
    with torch.no_grad():
        s = torch.FloatTensor(state).unsqueeze(0).to(device)
        return network(s).argmax().item()

class DQNAgent:
    def __init__(self, env, config):
        state_dim = env.observation_space.shape[0]
        n_actions = env.action_space.n
        self.n_actions   = n_actions
        self.gamma       = config.get('gamma', 0.99)
        self.batch_size  = config.get('batch_size', 64)
        self.eps_max     = config.get('eps_max', 1.0)
        self.eps_min     = config.get('eps_min', 0.05)
        self.eps_delay   = config.get('eps_delay', 100)
        self.eps_period  = config.get('eps_period', 5000)
        self.target_freq = config.get('target_update_freq', 200)
        self.epsilon     = self.eps_max
        self.eps_step    = (self.eps_max - self.eps_min) / self.eps_period

        self.model  = make_dqn(state_dim, n_actions)
        self.target = copy.deepcopy(self.model)
        self.memory = ReplayBuffer(config.get('buffer_size', 50_000), device)
        self.optim  = torch.optim.Adam(self.model.parameters(),
                                       lr=config.get('lr', 1e-3))
        self.loss_fn = nn.SmoothL1Loss()
        self.steps = 0

    def act(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return greedy_action(self.model, state)

    def learn_step(self):
        if len(self.memory) < self.batch_size:
            return
        s, a, r, s2, done = self.memory.sample(self.batch_size)
        with torch.no_grad():
            target_vals = r + self.gamma * (1 - done) * self.target(s2).max(1).values
        current_vals = self.model(s).gather(1, a.unsqueeze(1)).squeeze(1)
        loss = self.loss_fn(current_vals, target_vals)
        self.optim.zero_grad()
        loss.backward()
        self.optim.step()

    def train(self, env, n_episodes):
        episode_returns = []
        for ep in trange(n_episodes):
            state, _ = env.reset()
            ep_return = 0.0
            done = False
            while not done:
                action = self.act(state)
                next_state, reward, done, trunc, _ = env.step(action)
                self.memory.append(state, action, reward, next_state, float(done or trunc))
                self.learn_step()
                state = next_state
                ep_return += reward
                self.steps += 1
                # Epsilon decay
                if self.steps > self.eps_delay:
                    self.epsilon = max(self.eps_min, self.epsilon - self.eps_step)
                # Target network update
                if self.steps % self.target_freq == 0:
                    self.target.load_state_dict(self.model.state_dict())
                if trunc:
                    break
            episode_returns.append(ep_return)
        return episode_returns

In [ ]:
config = {
    'gamma': 0.99,
    'lr': 1e-3,
    'batch_size': 64,
    'buffer_size': 50_000,
    'eps_max': 1.0,
    'eps_min': 0.05,
    'eps_delay': 50,
    'eps_period': 3000,
    'target_update_freq': 200,
}

agent = DQNAgent(env_cp, config)
returns = agent.train(env_cp, n_episodes=300)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

def smooth(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(9, 3))
plt.plot(returns, alpha=0.3, label='episode return')
plt.plot(smooth(returns), label=f'{20}-ep moving avg')
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("DQN on CartPole-v1")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Final 50-episode average: {np.mean(returns[-50:]):.1f}")

In [ ]:
# Watch the trained agent
from gymnasium.utils.save_video import save_video
import os

test_env = gym.make("CartPole-v1", render_mode="rgb_array_list")
state, _ = test_env.reset()
for _ in range(500):
    action = greedy_action(agent.model, state)
    state, _, done, trunc, _ = test_env.step(action)
    if done or trunc:
        break

os.makedirs("videos", exist_ok=True)
save_video(test_env.render(), "videos",
           fps=test_env.metadata["render_fps"],
           name_prefix="dqn_cartpole")
test_env.close()

In [ ]:
from IPython.display import Video
Video("videos/dqn_cartpole-episode-0.mp4")

## DQN discussion

DQN works well when:
- The action space is **discrete and small** (the $\max_a$ operation is cheap).
- The state can be represented as a fixed-size vector (or image).

**What if the action space is continuous?**
FrozenLake has 4 actions; CartPole has 2.
The V2G problem has a **10-dimensional continuous action space** — taking the max over $a$ is no longer trivial.
→ This is where **actor-critic methods** come in.

# 5. Opening towards Soft Actor-Critic (SAC)

For **continuous action spaces**, we cannot enumerate $\max_a q^*(s, a)$.

**Actor-critic** methods maintain two networks:
- **Critic** $q_\phi(s, a)$: approximates the Q-function.
- **Actor** $\pi_\theta(a|s)$: outputs an action (or a distribution) directly.

The actor maximises the Q-function *implicitly* through gradient ascent:
$$\theta \leftarrow \theta + \alpha \nabla_\theta \mathbb{E}_{a \sim \pi_\theta(\cdot|s)}[q_\phi(s, a)].$$

**SAC** (Haarnoja et al., 2018) adds an **entropy bonus** to encourage exploration:
$$J(\pi) = \mathbb{E}\left[ \sum_t \gamma^t \left( r(s_t, a_t) + \alpha\, \mathcal{H}(\pi(\cdot|s_t)) \right) \right]$$

where $\mathcal{H}(\pi(\cdot|s)) = -\mathbb{E}_{a\sim\pi}[\log \pi(a|s)]$ is the policy **entropy** and $\alpha$ is a temperature parameter.

Key SAC ingredients:
- **Squashed Gaussian actor**: $a = \tanh(\mu + \sigma \epsilon)$, $\epsilon \sim \mathcal{N}(0,I)$.
- **Two critics** (to reduce overestimation bias).
- **Automatic entropy tuning**: $\alpha$ is adjusted so that $\mathcal{H}(\pi) \approx$ target entropy.

In practice, SAC is the go-to off-policy algorithm for continuous control and outperforms DQN on environments like V2G.

# 6. Back to V2G

<div class="alert alert-warning">

**Discussion and mini-exercise**
1. Can we apply tabular value iteration to V2G? Why or why not?
2. Can we apply DQN directly to V2G? What modification would be needed?
3. Which algorithm from this notebook seems most promising for V2G? Why?
</div>

In [ ]:
# Let's benchmark random vs ChargeAsFastAsPossible on V2G
from ev2gym.models.ev2gym_env import EV2Gym
from ev2gym.rl_agent.state import V2G_profit_max
from ev2gym.baselines.heuristics import ChargeAsFastAsPossible

env_v2g = EV2Gym(config_file="custom.yaml", save_replay=False,
                 save_plots=False, state_function=V2G_profit_max)

def eval_v2g(agent_fn, n_episodes=20):
    """Evaluate an agent on V2G; agent_fn(env) -> action array."""
    returns = []
    for _ in range(n_episodes):
        state, _ = env_v2g.reset()
        G = 0.0
        for _ in range(env_v2g.simulation_length):
            action = agent_fn(env_v2g)
            state, r, done, trunc, _ = env_v2g.step(action)
            G += r
            if done or trunc:
                break
        returns.append(G)
    return np.mean(returns), np.std(returns)

cafap = ChargeAsFastAsPossible()
mean_h, std_h = eval_v2g(lambda e: cafap.get_action(e))
mean_r, std_r = eval_v2g(lambda e: e.action_space.sample())

print(f"Heuristic (CAFAP): {mean_h:.2f} ± {std_h:.2f}")
print(f"Random policy    : {mean_r:.2f} ± {std_r:.2f}")
print()
print("An RL agent trained with SAC should outperform both.")
print("SAC is available via stable-baselines3:")
print("  from stable_baselines3 import SAC")
print("  model = SAC('MlpPolicy', env_v2g, verbose=1)")
print("  model.learn(total_timesteps=100_000)")

## Summary

| Method | State space | Action space | Key idea |
|---|---|---|---|
| Value Iteration | Discrete, tabular | Discrete | Exact Bellman contraction |
| DQN | Continuous | Discrete | Neural Q + replay + target net |
| SAC | Continuous | Continuous | Actor-critic + entropy reg. |

In the next notebook we will take a fundamentally different approach: instead of learning the value function and extracting a policy, we will **directly optimise the policy** using **policy gradient** methods.